# Consumption Audience
Пример отчета по расчету пересечения аудитории указанных ресурсов с другими медиаобъектами (Сross Web).
Расчет аналогичен расчету duplication, только в отличии от duplication выгружаются данные по пересечению указанных в mart_filter объектов и **сразу всех** остальных медиаобъектов указанных в slices через их тип, например "crossMediaResourceId". 

## Описание задачи

Посчитаем аудиторию по ресурсам:
- для пользователей Ozon - получим пересечение ресурсов и Ozon;
- для пользователей Ozon и Wildberries, которые посетили хотя бы один ресурс Ozon OR Wildberries - получим пересечение ресурсов и такого медиаобъекта как Ozon OR Wildberries;
- для пользователей Wildberries - получим пересечение ресурсов и Wildberries в разбивке на соцдем;

Общие параметры:
- Период: Июнь 2025
- География: Россия 0+
- Население: 12+
- Тип пользования интернетом: ограничения нет, считаем по всем (Web Desktop, Web Mobile, App Mobile)

Статистики:
- Reach (reach)
- Reach% (reachPer)

# Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к API Cross Web и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
import pandas as pd

from mediascope_api.crossweb import tasks as cwt
from mediascope_api.crossweb import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с API Cross Web
mtask = cwt.CrossWebTask()
cats = mtask.cats

In [ ]:
# список доступных статистик для отчета consumption media
cats.get_consumption_media_unit()['statistics']

In [ ]:
# список доступных срезов для отчета consumption media
cats.get_consumption_media_unit()['slices']

In [ ]:
# список доступных фильтров для отчета consumption media
cats.get_consumption_media_unit()['filters']

In [ ]:
# вывод справочника use type
cats.get_consumption_media_usetype()

## Формируем задания

Необходимо сформировать два задания для API Cross Web:
- расчет по пользователям аудитории ресурса Ozon
- расчет по пользователям аудитории Ozon OR Wildberries
- Wildberries в разбивке на соцдем

### Общие параметры для заданий

Для начала зададим общие параметры

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец'). Можно указать несколько периодов
date_filter = [('2025-06-01', '2025-06-30')]

# Задаем фильтр по типам пользования интернетом
usetype_filter = [1,2,3]

### Получим ID ресурсов
Для построения отчета необходимо получить идентификаторы ресурсов  __Ozon__ и __Wildberries__.

Для этого воспользуемся методами поиска в медиа справочнике. Работа с медиа справочником подробно описана в ноутбуке [catalogs](catalogs.ipynb). 

Получим идентификатор ресурсов   **Ozon**  **Wildberries**  

In [ ]:
cats.get_resource(resource='Wildberries')

Получим идентификатор ресурса **Ivi**

In [ ]:
cats.get_resource(resource='Ozon')

Таким образом, необходимые идентификаторы следующие:

- **Ozon**  resourceId = 1094
- **Wildberries**  resourceId = 3768

## Задания

Перейдем к формированию заданий.


### Задание №1. Расчет по аудитории посещающей ресурс Ozon

In [ ]:
# Задаем название для отображения в DataFrame
project_name = 'Ozon users'

# Задаем фильтр по географии, в нашем случае он не требуется
geo_filter = None

# Задаем фильтр по демографии, в нашем случае он не требуется
demo_filter = None

# Задаем фильтр по медиа, в нашем случае это ID ресурса Telegram
mart_filter = 'crossMediaResourceId = 1094'

# Указываем список срезов, чтобы сформировать структуру расчета, 
#если необходима разбивка на соцдем переменную - то добавляем ее название в кавычках "" через запятую
slices = ["researchMonth", "crossMediaResourceId"]

# Указываем список статистик для расчета
statistics = ['reach', 'reachPer'] 

# Формируем задание для API Cross Web в формате JSON
task_json = mtask.build_task(task_type='consumption-media', task_name=project_name, 
                             date_filter=date_filter, usetype_filter=usetype_filter, 
                             geo_filter=geo_filter, demo_filter=demo_filter, 
                             mart_filter=mart_filter, slices=slices, statistics=statistics)

# Отправляем задание на расчет и ждем выполнения
task = mtask.wait_task(mtask.send_consumption_media_task(task_json))

# Получаем результат
df_first = mtask.result2table(mtask.get_result(task), project_name = project_name)
df_first

### Задание №2. Расчет по совокупной аудитории посещающей ресурсы Ozon или Wildberries

In [ ]:
# Задаем название для отображения в DataFrame
project_name = 'Ozon + Wildberries users'

# Задаем фильтр по географии, в нашем случае он не требуется
geo_filter = None

# Задаем фильтр по демографии, в нашем случае он не требуется
demo_filter = None

# Задаем фильтр по медиа, в нашем случае это ID ресурсов Ozon или Wildberries
mart_filter = 'crossMediaResourceId in (1094, 3768)'

# Указываем список срезов, чтобы сформировать структуру расчета
slices = ["researchMonth", "crossMediaResourceId"]

# Указываем список статистик для расчета
statistics = ['reach', 'reachPer']

# Формируем задание для API Cross Web в формате JSON
task_json = mtask.build_task(task_type='consumption-media', task_name=project_name, 
                             date_filter=date_filter, usetype_filter=usetype_filter, 
                             geo_filter=geo_filter, demo_filter=demo_filter, 
                             mart_filter=mart_filter, slices=slices, statistics=statistics)

# Отправляем задание на расчет и ждем выполнения
task = mtask.wait_task(mtask.send_consumption_media_task(task_json))

# Получаем результат
df_second = mtask.result2table(mtask.get_result(task), project_name)
df_second

### Задание №3. Расчет по аудитории посещающей Wildberries в разбивке на соцдем

In [ ]:
# Задаем название для отображения в DataFrame
project_name = 'Wildberries users by demo'

# Задаем фильтр по географии, в нашем случае он не требуется
geo_filter = None

# Задаем фильтр по демографии, в нашем случае он не требуется
demo_filter = None

# Задаем фильтр по медиа, в нашем случае это ID ресурса Telegram
mart_filter = 'crossMediaResourceId = 3768'

# Указываем список срезов, чтобы сформировать структуру расчета, 
#если необходима разбивка на соцдем переменную - то добавляем ее название в кавычках "" через запятую
slices = ["researchMonth", "crossMediaResourceId", "sex"]

# Указываем список статистик для расчета
statistics = ['reach', 'reachPer'] 

# Формируем задание для API Cross Web в формате JSON
task_json = mtask.build_task(task_type='consumption-media', task_name=project_name, 
                             date_filter=date_filter, usetype_filter=usetype_filter, 
                             geo_filter=geo_filter, demo_filter=demo_filter, 
                             mart_filter=mart_filter, slices=slices, statistics=statistics)

# Отправляем задание на расчет и ждем выполнения
task = mtask.wait_task(mtask.send_consumption_media_task(task_json))

# Получаем результат
df_third = mtask.result2table(mtask.get_result(task), project_name = project_name)
df_third

### Формирование итоговой таблицы

Следующим шагом соберем полученные результаты в один DataFrame

In [ ]:
# Объединим наши DataFrame'ы
df_result = pd.concat([df_first, df_second, df_third])
df_result

## Сохраняем в Excel

In [ ]:
df_info = mtask.get_report_info()

with pd.ExcelWriter(mtask.get_excel_filename('consumption-media-ozon-wildberries')) as writer:
    df_result.to_excel(writer, sheet_name='Report', index=False)
    df_info.to_excel(writer, sheet_name='Info', index=False)